In [1]:
import os
if os.getcwd().endswith('scripts'):
    os.chdir('..')
print(f"Current working directory: {os.getcwd()}")

Current working directory: /data6/yuzihan/WorkSpace/35-HuaweiCrowdSimulation/HuaweiCrowdSimulationCode


In [2]:
from src.dataset import ETHDataset, UCYDataset, SDDDataset, GCDataset, WayMoDataset
from argparse import Namespace
from src.utils.logger import init_logger

init_logger('src')

args = Namespace(
    name="train",
    exp_name="20251009_train_122049_LM2",
    device="cuda:0",
    batch_size=256,
    lr=0.001,
    epochs=10000,
    patience=20,
    sampling_method="DDIM",
    T=100,
    sample_num=1,
    denoise_step=5,
    hist_step=8,
    pred_step=1,
    skip_step=1,
    roll_step=12,
    fps=2.5,
    dot_per_meter=5,
    seed=550,
    save_dir="./logs/train",
    debug=False,
    model_dim=128,
    map_feature_dim=64,
    head_num=4,
    dropout=0.3,
    latent_token_num=16,
    beta_schedule="cosine",
    num_workers=0,
    datasets="ETH/UCY",
    test_name=None,
    test_ratio=None,
    split_by_scenario=False,
    cache_dataset=True,
    test_before_train=True,
    test_per_epoch=10,
    save_per_epoch=50,
    reload_checkpoint=None,
    save_path="logs/train/20251009_train_122049_LM2",
)

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

dataset_list = [
    *ETHDataset.load_data_batch(args, "./data/ETH/"),
    *UCYDataset.load_data_batch(args, "./data/UCY/"),
    GCDataset.load_data(args, "./data/GC/Annotation"),
    *SDDDataset.load_data_batch(args, "./data/SDD/"),
]

df_list = []
for dataset in tqdm(dataset_list):
    df_data = dataset.df_data
    map_data = dataset.map_data
    df_data = df_data[df_data['type'] == 'pedestrian']
    traj_length = np.mean([traj[['x', 'y']].diff().pow(2).sum(axis=1).pow(0.5).sum() for id, traj in df_data.groupby('id')])
    traj_duration = np.mean([(traj['f'].max() - traj['f'].min()) / args.fps for id, traj in df_data.groupby('id')])

    row = {
        'dataset': type(dataset).__name__.removesuffix('Dataset'),
        '#Trajectory': df_data['id'].nunique(),
        '#Frame': df_data['f'].nunique(),
        'Time': (df_data['f'].max() - df_data['f'].min()) / args.fps,
        'Area': (map_data.xmax - map_data.xmin) * (map_data.ymax - map_data.ymin),
        'Speed': df_data[['x', 'y']].diff().pow(2).sum(axis=1).pow(0.5).mean() * args.fps,
        'Trajectory-Length': traj_length,
        'Trajectory-Duration': traj_duration,
    }
    df_list.append(row)

df = pd.DataFrame(df_list)

df.groupby('dataset').agg(**{
    '#Scenario': ('#Trajectory', 'count'),
    '#Pedestrian': ('#Trajectory', 'sum'),
    '#Frame': ('#Frame', 'sum'),
    'Time': ('Time', 'sum'),
    'Average Area': ('Area', 'mean'),
    'Average Trajectory-Length': ('Trajectory-Length', 'mean'),
    'Average Trajectory-Duration': ('Trajectory-Duration', 'mean'),
    'Average Speed': ('Speed', 'mean'),
    # 'Area': ('Area', 'sum'),
    # 'Trajectory-Length': ('Trajectory-Length', 'sum'),
    # 'Trajectory-Duration': ('Trajectory-Duration', 'sum'),
    # 'Average Time': ('Time', 'mean'),
})

[None|eth_dataset|I|Oct09 14:20:28|0:13:35.057595] Loading cached dataset-list from data/.cache/ETH.pkl
Loading ETH datasets: 100%|██████████| 2/2 [00:00<00:00, 44.42it/s, seq_hotel]
[None|ucy_dataset|I|Oct09 14:20:28|0:13:35.108203] Caching dataset-list to data/.cache/UCY.pkl
Loading UCY datasets: 100%|██████████| 7/7 [00:00<00:00, 61.24it/s, data_zara/crowds_zara03]
[None|gc_dataset|I|Oct09 14:20:28|0:13:35.225657] Loading cached dataset from data/.cache/GC_2.5_8_1_1.pkl


,#Scenario,#Pedestrian,#Frame,Time,Average Area,Average Trajectory-Length,Average Trajectory-Duration,Average Speed
dataset,,,,,,,,
ETH,2,749,2016,1186.4,490.838498,9.455978,5.785079,2.864477
GC,1,12675,12000,4799.6,3706.106396,33.946065,38.865862,1.588309
UCY,7,1480,4468,1801.6,341.527181,14.959619,16.195875,1.723080


In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm

args.cache_dataset = False
dataset_list = [
    # *ETHDataset.load_data_batch(args, "./data/ETH/"),
    # *UCYDataset.load_data_batch(args, "./data/UCY/"),
    # GCDataset.load_data(args, "./data/GC/Annotation"),
    # *SDDDataset.load_data_batch(args, "./data/SDD/"),
    *WayMoDataset.load_data_batch(args, "./data/WayMo/Processed"),
]

df_list = []
for dataset in tqdm(dataset_list):
    df_data = dataset.df_data
    map_data = dataset.map_data
    df_data = df_data[df_data['type'] == 'pedestrian']
    traj_length = np.mean([traj[['x', 'y']].diff().pow(2).sum(axis=1).pow(0.5).sum() for id, traj in df_data.groupby('id')])
    traj_duration = np.mean([(traj['f'].max() - traj['f'].min()) / args.fps for id, traj in df_data.groupby('id')])

    row = {
        'dataset': type(dataset).__name__.removesuffix('Dataset'),
        '#Trajectory': df_data['id'].nunique(),
        '#Frame': df_data['f'].nunique(),
        'Time': (df_data['f'].max() - df_data['f'].min()) / args.fps,
        'Area': (map_data.xmax - map_data.xmin) * (map_data.ymax - map_data.ymin),
        'Speed': df_data[['x', 'y']].diff().pow(2).sum(axis=1).pow(0.5).mean() * args.fps,
        'Trajectory-Length': traj_length,
        'Trajectory-Duration': traj_duration,
    }
    df_list.append(row)

df = pd.DataFrame(df_list)

df.groupby('dataset').agg(**{
    '#Scenario': ('#Trajectory', 'count'),
    '#Pedestrian': ('#Trajectory', 'sum'),
    '#Frame': ('#Frame', 'sum'),
    'Time': ('Time', 'sum'),
    'Average Area': ('Area', 'mean'),
    'Average Trajectory-Length': ('Trajectory-Length', 'mean'),
    'Average Trajectory-Duration': ('Trajectory-Duration', 'mean'),
    'Average Speed': ('Speed', 'mean'),
    # 'Area': ('Area', 'sum'),
    # 'Trajectory-Length': ('Trajectory-Length', 'sum'),
    # 'Trajectory-Duration': ('Trajectory-Duration', 'sum'),
    # 'Average Time': ('Time', 'mean'),
})

[None|waymo_dataset|I|Oct09 14:39:58|0:00:22.874223] Caching dataset-list to data/.cache/WayMo-Processed.pkl
Loading SDD datasets:   0%|          | 0/1788 [00:00<?, ?it/s, Processed/00001_2aa43fad083efbf3][None|base_dataset|I|Oct09 14:39:58|0:00:23.361409] Normalized x with mean=125.4211, std=1.0000
[None|base_dataset|I|Oct09 14:39:58|0:00:23.362507] Normalized y with mean=700.9705, std=1.0000
[None|base_dataset|I|Oct09 14:39:58|0:00:23.364026] Normalized map into xmin=-91.1004, xmax=-48.1998, ymin=75.5691, ymax=104.2940
100%|██████████| 30/30 [00:00<00:00, 33.32it/s]
[None|waymo_dataset|I|Oct09 14:39:59|0:00:24.272919] Caching dataset to data/.cache/00001_2aa43fad083efbf3_2.5_8_1_1.pkl
Loading SDD datasets:   0%|          | 1/1788 [00:01<41:32,  1.39s/it, Processed/00004_e2030d66ebfe7b6b][None|base_dataset|I|Oct09 14:39:59|0:00:24.649467] Normalized x with mean=-831.6093, std=1.0000
[None|base_dataset|I|Oct09 14:39:59|0:00:24.651490] Normalized y with mean=-697.3996, std=1.0000
[None|